<a href="https://colab.research.google.com/github/dsouzabrandon02-ops/CSL701-14_Machine-Learning-Lab/blob/main/Task_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Name : Brandon .B. Dsouza
Roll No : CS14



In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
column_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=column_names)

display(df.head())

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:

anomalous_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in anomalous_cols:
    df[col] = df[col].replace(0, np.nan)
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

X = df.drop(columns=['Outcome']).values
y = df['Outcome'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (614, 8)
Testing set shape: (154, 8)


### 3. Implement Custom Bootstrap Function & Train Base Learners
We will write a helper function to perform bootstrap sampling (sampling with replacement) and train $B=15$ unpruned `DecisionTreeClassifier` estimators.

In [3]:
def bootstrap_sample(X, y):
    n_samples = X.shape[0]

    indices = np.random.choice(n_samples, size=n_samples, replace=True)
    return X[indices], y[indices]

B = 15
base_estimators = []

np.random.seed(42)

for i in range(B):
    X_b, y_b = bootstrap_sample(X_train, y_train)
    clf = DecisionTreeClassifier(random_state=42 + i)
    clf.fit(X_b, y_b)
    base_estimators.append(clf)

print(f"Successfully trained {len(base_estimators)} base Decision Tree estimators.")

Successfully trained 15 base Decision Tree estimators.


### 4. Custom Aggregation with Majority Voting
Now we write a custom prediction function that aggregates predictions from all base estimators using majority voting.

In [4]:
def predict_bagging(estimators, X):

    predictions = np.array([clf.predict(X) for clf in estimators])


    predictions = predictions.T


    final_preds = []
    for row in predictions:

        classes, counts = np.unique(row, return_counts=True)
        majority_class = classes[np.argmax(counts)]
        final_preds.append(majority_class)

    return np.array(final_preds)

y_pred_bagging = predict_bagging(base_estimators, X_test)
bagging_acc = accuracy_score(y_test, y_pred_bagging)
bagging_cm = confusion_matrix(y_test, y_pred_bagging)

print(f"Custom Bagging Test Accuracy: {bagging_acc:.4f}")

Custom Bagging Test Accuracy: 0.7597


### 5. Compare with Standalone Single Decision Tree
Finally, we will train a standard, single unpruned `DecisionTreeClassifier` on the full training set and compare its performance with our custom Bagging model.

In [5]:

single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)
y_pred_single = single_tree.predict(X_test)

single_acc = accuracy_score(y_test, y_pred_single)
single_cm = confusion_matrix(y_test, y_pred_single)

print("=== Performance Comparison ===")
print(f"Single Decision Tree Accuracy : {single_acc:.4f}")
print(f"Custom Bagging Accuracy       : {bagging_acc:.4f}\n")

print("Single Decision Tree Confusion Matrix:")
print(single_cm)
print("\nCustom Bagging Confusion Matrix:")
print(bagging_cm)

=== Performance Comparison ===
Single Decision Tree Accuracy : 0.7208
Custom Bagging Accuracy       : 0.7597

Single Decision Tree Confusion Matrix:
[[76 23]
 [20 35]]

Custom Bagging Confusion Matrix:
[[75 24]
 [13 42]]
